# 05 — v4 (decoder concat injection)

v4 concatenates the raw source/candidate nutrient vectors at the decoder. Single train + test at the optimal `LAMBDA_H`.

**Runtime**: Colab free T4 GPU, ~30 min.

**How to run**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell below) if your Drive layout differs
3. (optional) Edit the **Config** cell to change `LAMBDA_H` / `TAU_PERCENTILE`
4. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# Versions follow requirements.txt; torch ships with Colab. torch_geometric
# must match the installed torch build — if its import fails later, install
# the PyG wheel for the torch version printed here:
#   https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
import torch; print('torch', torch.__version__)
!pip install -q 'torch_geometric>=2.4' 'pandas>=2.0' 'numpy>=1.24' matplotlib

## 2. Mount Drive (code + data both live in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR     = f'{PROJECT_ROOT}/code'
DATA_DIR     = f'{PROJECT_ROOT}/data'
# Final-submission runs live under outputs/final so they never
# clobber the exploratory sweep outputs in outputs/.
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs/final'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'CWD        = {os.getcwd()}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# Fail fast with a clear message if the dataset isn't fully prepared.
# Training needs the substitution pairs + recipes on top of the graph;
# produce them with src/convert_data.py (or notebooks/01_setup_data.ipynb).
_required = ['flavorgraph_edges.csv', 'nodes_filtered.csv', 'usda_mapping.json',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv', 'recipes.json']
_missing = [f for f in _required if not os.path.exists(f'{DATA_DIR}/{f}')]
if _missing:
    raise FileNotFoundError(
        f'Missing data files in {DATA_DIR}: {_missing}. Prepare the dataset '
        'first (src/convert_data.py or notebooks/01_setup_data.ipynb).')
print('[data] all required files present')

## 3. Config — hyper-parameters

The only knobs worth changing. `LAMBDA_H` weights `L_health` against the
GISMo substitution loss; `TAU_PERCENTILE` sets the goal-threshold percentile
(0 = "any reduction counts"). `LAMBDA_H = 1.0` is used for every model in the report.

In [ ]:
LAMBDA_H       = 1.0   # health-loss weight (same for all models)
TAU_PERCENTILE = 0     # goal threshold percentile (0 = any reduction)
print(f'LAMBDA_H = {LAMBDA_H},  TAU_PERCENTILE = {TAU_PERCENTILE}')

## 4. Smoke test (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Set RUN_SMOKE = False to skip.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v4.py --lambda_h {LAMBDA_H} --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v4
    print('\n[smoke] OK')

## 5. Train + test v4

All four `g` overrides (`auto`, `1_0`, `0_1`, `1_1`) are evaluated from the single trained checkpoint so the test cell below can show the g-sensitivity of the model (does it actually use the goal vector?).

In [ ]:
!python src/train_v4.py --lambda_h {LAMBDA_H} --tau_percentile {TAU_PERCENTILE} \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v4

print('\n[done] checkpoint:', f'{OUTPUT_DIR}/v4/best_v4.pt')

## 6. Test metrics (+ g-override sensitivity)

If the model uses `g`, the low-sugar goal `g=1_0` should give higher `sugar_sat` than the low-sodium goal `g=0_1`, and vice versa for sodium.

In [ ]:
import os, json, sys, contextlib, io
import pandas as pd

sys.path.insert(0, f'{CODE_DIR}/src')
from evaluate_health import compute_health_metrics
from evaluate_flavor import evaluate_flavor
from evaluate_id_ood import evaluate_id_ood


def _quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def collect_metrics(pred_path):
    if not os.path.exists(pred_path):
        return None
    with open(pred_path) as f:
        pred = json.load(f)
    if 'metrics' not in pred:
        print(f'  [skip] {pred_path}: no metrics block')
        return None
    # Tolerate a missing key (older/partial file) — drop just that column.
    out = {k: pred['metrics'][k] for k in ('MRR', 'Hit@1', 'Hit@3', 'Hit@10')
           if k in pred['metrics']}
    try:
        h = _quiet(compute_health_metrics, pred_path,
                   f'{DATA_DIR}/usda_mapping.json', save_to=None)
        out['sugar_sat']   = h['pred']['sugar_g']['satisfaction_rate']
        out['sodium_sat']  = h['pred']['sodium_mg']['satisfaction_rate']
        out['d_sugar_g']   = h['pred']['sugar_g']['avg_delta']
        out['d_sodium_mg'] = h['pred']['sodium_mg']['avg_delta']
    except Exception as e:
        print(f'  [health failed] {e}')
    try:
        fv = _quiet(evaluate_flavor, pred_path,
                    f'{DATA_DIR}/flavorgraph_edges.csv', save_to=None)
        out['flavor_cos'] = fv['pred_cosine']['mean']
    except Exception as e:
        print(f'  [flavor failed] {e}')
    try:
        io_ = _quiet(evaluate_id_ood, pred_path,
                     f'{DATA_DIR}/pairs_train.csv', save_to=None)
        out['ID_MRR']  = io_['id']['MRR']
        out['OOD_MRR'] = io_['ood']['MRR']
    except Exception as e:
        print(f'  [idood failed] {e}')
    return out


def show_table(path_dict):
    rows = []
    for label, path in path_dict.items():
        m = collect_metrics(path)
        if m is None:
            print(f'  skip [{label}]: not found  {path}')
            continue
        rows.append({'config': label, **m})
    if not rows:
        print('(no predictions found)')
        return None
    df = pd.DataFrame(rows).set_index('config')
    cols = ['MRR', 'Hit@1', 'Hit@10', 'sugar_sat', 'sodium_sat',
            'flavor_cos', 'ID_MRR', 'OOD_MRR']
    print(df[[c for c in cols if c in df.columns]].round(2).to_string())
    return df

In [ ]:
print(f'=== v4 (lambda_h={LAMBDA_H}) — test, all g overrides ===')
df = show_table({
    'auto':            f'{OUTPUT_DIR}/v4/test_predictions_v4_auto.json',
    '1_0 (low sugar)':  f'{OUTPUT_DIR}/v4/test_predictions_v4_1_0.json',
    '0_1 (low sodium)': f'{OUTPUT_DIR}/v4/test_predictions_v4_0_1.json',
    '1_1 (both)':       f'{OUTPUT_DIR}/v4/test_predictions_v4_1_1.json',
})

ok = (df is not None
      and {'1_0 (low sugar)', '0_1 (low sodium)'}.issubset(df.index)
      and {'sugar_sat', 'sodium_sat'}.issubset(df.columns))
if ok:
    s10 = df.loc['1_0 (low sugar)', 'sugar_sat']
    s01 = df.loc['0_1 (low sodium)', 'sugar_sat']
    n10 = df.loc['1_0 (low sugar)', 'sodium_sat']
    n01 = df.loc['0_1 (low sodium)', 'sodium_sat']
    # Each line: targeted-goal value first, other-goal second; effect = first - second
    # (positive => the goal vector g actually shifts predictions the intended way).
    print(f'\n  sugar_sat:  low-sugar g=1_0 -> {s10:.1f}  vs  low-sodium g=0_1 -> {s01:.1f}   (g effect {s10-s01:+.1f}pp)')
    print(f'  sodium_sat: low-sodium g=0_1 -> {n01:.1f}  vs  low-sugar g=1_0 -> {n10:.1f}   (g effect {n01-n10:+.1f}pp)')